# Random Agent Baseline

Evaluates a purely random Minesweeper agent (uniform over valid hidden cells) on the beginner preset (9×9, 10 mines). Produces performance metrics, plots, and a recorded episode GIF.

This is the **floor baseline** that any learned Q/DQN agent must beat.

In [ ]:
# --- headless pygame MUST be set before any pygame import ---
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "imageio"], check=True)

sys.path.insert(0, "..")

import random
import numpy as np
import matplotlib.pyplot as plt
import imageio
import pygame
from IPython.display import Image as IPImage, display

from src.env.minesweeper_env import MinesweeperEnv

print(f"pygame {pygame.version.ver}")
print("All imports OK")

## Random Agent

In [ ]:
class RandomAgent:
    """
    Selects uniformly at random from valid (hidden) cells.
    Never wastes a move on an already-revealed cell.
    """
    def __init__(self, rng_seed: int = None):
        self._rng = random.Random(rng_seed)

    def choose_action(self, env: MinesweeperEnv) -> int:
        return self._rng.choice(env.get_valid_actions())

## Evaluation (1000 episodes)

In [ ]:
N_EPISODES = 1000
PRESET = "beginner"  # 9x9, 10 mines, 71 safe cells
TOTAL_SAFE = 9 * 9 - 10  # 71

episode_lengths   = []
outcomes          = []
total_rewards     = []
cells_revealed    = []
first_move_deaths = []

agent = RandomAgent(rng_seed=0)

for ep in range(N_EPISODES):
    env = MinesweeperEnv(preset=PRESET, seed=ep)
    obs = env.reset()

    moves = 0
    ep_reward = 0.0
    done = False
    final_info = None

    while not done:
        action = agent.choose_action(env)
        obs, reward, done, info = env.step(action)
        moves += 1
        ep_reward += reward
        final_info = info

    # Count safe cells revealed: obs values 0-8 only (excludes MINE_EXPOSED=10)
    n_safe = int(((obs >= 0) & (obs <= 8)).sum())

    episode_lengths.append(moves)
    outcomes.append(final_info["state"])
    total_rewards.append(ep_reward)
    cells_revealed.append(n_safe)
    # First-move death is structurally impossible (lazy mine placement guarantees
    # the first reveal is always safe), but we track it as an env sanity check.
    first_move_deaths.append(moves == 1 and final_info["state"] == "LOST")

print(f"Done. {N_EPISODES} episodes evaluated.")

## Summary Statistics

In [ ]:
wins   = sum(1 for o in outcomes if o == "WON")
losses = N_EPISODES - wins
win_rate = wins / N_EPISODES
rwd_per_move = np.array(total_rewards) / np.array(episode_lengths)

print("=" * 56)
print(f"  Random Agent  —  Beginner Minesweeper  ({N_EPISODES} episodes)")
print("=" * 56)
print(f"  Win rate:                {win_rate*100:6.2f}%  ({wins}/{N_EPISODES})")
print(f"  First-move death rate:   {sum(first_move_deaths)/N_EPISODES*100:6.2f}%  (expected 0 — env invariant)")
print()
print(f"  Episode length  mean:    {np.mean(episode_lengths):7.2f}  moves")
print(f"                  median:  {np.median(episode_lengths):7.2f}  moves")
print(f"                  std:     {np.std(episode_lengths):7.2f}  moves")
print(f"                  max:     {max(episode_lengths):7d}  moves")
print()
print(f"  Cells revealed  mean:    {np.mean(cells_revealed):7.2f}  / {TOTAL_SAFE}")
print(f"                  median:  {np.median(cells_revealed):7.2f}")
print(f"                  std:     {np.std(cells_revealed):7.2f}")
print()
print(f"  Total reward    mean:    {np.mean(total_rewards):8.2f}")
print(f"                  std:     {np.std(total_rewards):8.2f}")
print(f"  Reward/move     mean:    {np.mean(rwd_per_move):8.4f}")
print()
print("  --- Q-Learning Baseline (floors to beat) ---")
print(f"  Win rate floor:          {win_rate*100:6.2f}%")
print(f"  Avg cells revealed/ep:   {np.mean(cells_revealed):7.2f}  / {TOTAL_SAFE}")
print(f"  Avg reward/move:         {np.mean(rwd_per_move):8.4f}")
print("=" * 56)

## Plots

In [ ]:
os.makedirs("../recordings", exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Random Agent — Beginner Minesweeper (1000 Episodes)", fontsize=14)

wins_arr = np.array([1.0 if o == "WON" else 0.0 for o in outcomes])

# 1 — Rolling win rate
ax = axes[0, 0]
window = 50
rolling_wr = np.convolve(wins_arr, np.ones(window) / window, mode="valid")
ax.plot(range(window - 1, N_EPISODES), rolling_wr * 100, color="steelblue", lw=1.5)
ax.axhline(win_rate * 100, color="orange", lw=1.2, ls="--",
           label=f"Overall: {win_rate*100:.1f}%")
ax.set_title(f"Win Rate (rolling {window}-ep window)")
ax.set_xlabel("Episode")
ax.set_ylabel("Win rate (%)")
ax.set_ylim(0, max(rolling_wr * 100) * 2.5 + 1)
ax.legend()
ax.grid(True, alpha=0.3)

# 2 — Episode length histogram
ax = axes[0, 1]
ax.hist(episode_lengths, bins=30, color="teal", edgecolor="white", alpha=0.85)
ax.axvline(np.mean(episode_lengths), color="orange", lw=1.5, ls="--",
           label=f"Mean: {np.mean(episode_lengths):.1f}")
ax.set_title("Episode Length Distribution")
ax.set_xlabel("Number of moves")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.3)

# 3 — Cells revealed histogram
ax = axes[0, 2]
ax.hist(cells_revealed, bins=30, color="mediumseagreen", edgecolor="white", alpha=0.85)
ax.axvline(np.mean(cells_revealed), color="orange", lw=1.5, ls="--",
           label=f"Mean: {np.mean(cells_revealed):.1f}")
ax.axvline(TOTAL_SAFE, color="red", lw=1.2, ls=":", label=f"Max: {TOTAL_SAFE}")
ax.set_title("Safe Cells Revealed per Episode")
ax.set_xlabel("Cells revealed")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.3)

# 4 — Total reward histogram
ax = axes[1, 0]
ax.hist(total_rewards, bins=35, color="mediumpurple", edgecolor="white", alpha=0.85)
ax.axvline(np.mean(total_rewards), color="orange", lw=1.5, ls="--",
           label=f"Mean: {np.mean(total_rewards):.1f}")
ax.set_title("Total Episode Reward")
ax.set_xlabel("Total reward")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.3)

# 5 — WON / LOST bar chart
ax = axes[1, 1]
bars = ax.bar(["WON", "LOST"], [wins, losses],
              color=["steelblue", "tomato"], edgecolor="white", width=0.5)
for bar, cnt in zip(bars, [wins, losses]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f"{cnt}\n({cnt/N_EPISODES*100:.1f}%)",
            ha="center", va="bottom", fontsize=11)
ax.set_title("Episode Outcomes")
ax.set_ylabel("Count")
ax.set_ylim(0, N_EPISODES * 1.15)
ax.grid(True, alpha=0.3, axis="y")

# 6 — Reward per move histogram
ax = axes[1, 2]
ax.hist(rwd_per_move, bins=30, color="coral", edgecolor="white", alpha=0.85)
ax.axvline(np.mean(rwd_per_move), color="navy", lw=1.5, ls="--",
           label=f"Mean: {np.mean(rwd_per_move):.3f}")
ax.set_title("Reward per Move")
ax.set_xlabel("Reward / move")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../recordings/random_agent_stats.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: recordings/random_agent_stats.png")

## Record One Episode as GIF

In [ ]:
from src.game.renderer import Renderer

GIF_PATH = "../recordings/random_agent_episode.gif"
RECORD_SEED = 7

pygame.init()

rec_env = MinesweeperEnv(preset=PRESET, seed=RECORD_SEED)
obs = rec_env.reset()

# Renderer takes the Game object — env stores it as _game
rec_renderer = Renderer(rec_env._game, preset=PRESET)

frames = []

def capture():
    rec_renderer._draw()
    return pygame.surfarray.array3d(rec_renderer.screen).transpose(1, 0, 2).copy()

frames.append(capture())  # initial board state

rec_agent = RandomAgent(rng_seed=RECORD_SEED)
done = False
rec_moves = 0

while not done:
    action = rec_agent.choose_action(rec_env)
    obs, reward, done, info = rec_env.step(action)
    rec_moves += 1
    frames.append(capture())

print(f"Episode: {rec_moves} moves, outcome={info['state']}")
print(f"Captured {len(frames)} frames")

imageio.mimsave(GIF_PATH, frames, duration=0.3, loop=0)
print(f"Saved: {GIF_PATH}")

display(IPImage(filename=GIF_PATH))

## Q-Learning Baseline Interpretation

| Metric | Random Agent | Q-Learning Target |
|--------|-------------|-------------------|
| Win rate | see above | must exceed random |
| Avg cells revealed | see above | higher → better board coverage |
| Avg reward/move | see above | push toward positive |
| Episode length | see above | longer → surviving more |

**Notes for training:**

- **Win rate** is the primary goal. A random agent on beginner gets ~3–5% (first click is always safe due to lazy mine placement; all subsequent clicks are blind guesses).
- **First-move death = 0%** confirms the env's safe-first-click invariant: `Board.place_mines()` runs after the first reveal, excluding that cell. Any learned policy should inherit this free guarantee.
- **Reward/move is negative** for the random agent — mine penalties (−10) outweigh average reveal gains. A Q-agent that learns to avoid mine-adjacent cells should push this positive.
- **Rolling win rate is flat** — confirms no learning signal, i.e., this is a pure i.i.d. policy. Contrast with a trained agent's rolling curve trending upward over training.
- **Cells revealed distribution** is bimodal for random play: many short episodes (die quickly on densely mined areas) and a few long episodes (lucky open regions). A trained agent should shift the distribution right.
- **Episode length max ≤ 71** — you can't take more moves than safe cells exist on a 9×9 board with 10 mines (71 safe cells). A win always ends at exactly the number of safe cells revealed.